# Pulsed LFM and FMCW range processing

This notebook accompanies [SignalDetect.py](SignalDetect.py) and [pulse_compress.md](pulse_compress.md). It shows the figures disabled by `if 0` in the script, then compares pulse compression with FMCW range processing. Edit the parameters and choose **Run All**.

## Contents

- [Chirp and parameters](#chirp-and-parameters)
- [Received pulse](#received-pulse)
- [Received FFT and spectrogram](#received-fft-and-spectrogram)
- [Matched filtering and range](#matched-filtering-and-range)
- [FMCW dechirping](#fmcw-dechirping)
- [FMCW spectrum and range](#fmcw-spectrum-and-range)
- [Comparison and limitations](#comparison-and-limitations)

## Chirp and parameters

An LFM chirp sweeps frequency linearly with slope $S=B/T$. Its phase is the integral of frequency. The complex baseband signal used here is

$$s_{\mathrm{TX}}(t)=e^{j\pi S t^2},\qquad f_{\mathrm{TX}}(t)=St,\qquad 0\leq t<T.$$

The 9 GHz carrier is not directly sampled. The nominal bandwidth-limited range resolution is $\Delta R\approx c/(2B)$; the sample-delay grid spacing $c/(2F_s)$ is a separate quantity.

In [ ]:
import os
import sys
from pathlib import Path

# Make native libraries available when a Windows conda kernel is not activated.
library_bin = Path(sys.prefix) / 'Library' / 'bin'
if sys.platform == 'win32' and (Path(sys.prefix) / 'conda-meta').is_dir() and library_bin.is_dir():
    os.environ['PATH'] = str(library_bin) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve, find_peaks, spectrogram

# Parameters from SignalDetect.py; change these and run all cells again.
c = 3e8
f0 = 9e9              # RF carrier, not sampled in this baseband model
B = 100e6             # swept bandwidth (Hz)
T = 20e-6             # chirp duration (s)
Fs = 200e6            # sample rate (Hz)
targets = np.array([50.0, 80.0, 120.0])  # ranges (m)
amplitudes = np.array([1.0, 0.8, 0.7])

if B <= 0 or T <= 0 or Fs <= 0 or len(targets) == 0 or len(targets) != len(amplitudes) or np.any(targets < 0):
    raise ValueError('Use positive B, T and Fs, and matching nonempty arrays of non-negative target ranges and amplitudes.')
S = B / T
N = int(Fs * T)
if N < 256:
    raise ValueError('This spectrogram requires at least 256 chirp samples.')
t = np.arange(N) / Fs
tx = np.exp(1j * np.pi * S * t**2)
print(f'Slope: {S / 1e12:g} THz/s; samples: {N}')
print(f'Nominal range resolution: {c / (2 * B):g} m; delay-grid spacing: {c / (2 * Fs):g} m')

## Received pulse

A target at range $R_i$ returns a delayed copy with round-trip delay $\tau_i=2R_i/c$. The pulsed model rounds each delay to the nearest sample, $d_i=\operatorname{round}(F_s\tau_i)$, and adds the echoes coherently:

$$r[n]=\sum_i A_i\,s_{\mathrm{TX}}[n-d_i].$$

The receive buffer includes the entire latest echo. The diagrams are illustrative and may use different parameters from the simulation.

![Transmit and delayed receive pulse](Compress/b.svg)

![Example of delayed samples from multiple targets](Compress/c.svg)

In [ ]:
delays = 2 * targets / c
delay_samples = np.round(delays * Fs).astype(int)
rx_length = N + delay_samples.max() + 1
rx = np.zeros(rx_length, dtype=complex)
for delay, amplitude in zip(delay_samples, amplitudes):
    rx[delay:delay + N] += amplitude * tx

for distance, delay, samples in zip(targets, delays, delay_samples):
    print(f'{distance:g} m: delay {delay * 1e6:.3f} us, rounded to {samples} samples')

# The originally disabled plot_rx figure.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(len(rx)) / Fs * 1e6, np.abs(rx))
ax.set(xlabel='Time (us)', ylabel='Amplitude', title='Received pulsed LFM signal')
ax.grid(True)
fig.tight_layout()
plt.show()

## Received FFT and spectrogram

The received signal is complex, so its FFT has both positive and negative baseband frequencies. The FFT summarizes the whole pulse; the spectrogram uses overlapping windows to show frequency versus time.

For both spectral magnitudes we plot **dB relative to each plot's own peak**, not dBm or absolute power:

$$L_{\mathrm{dB}}=20\log_{10}\!\left(\frac{|X|}{\max |X|}\right).$$

A -120 dB floor prevents taking the logarithm of zero. It is a display floor, not a noise measurement.

In [ ]:
def relative_db(magnitude):
    magnitude = np.abs(magnitude)
    peak = np.max(magnitude)
    if not np.isfinite(peak) or peak <= 0:
        raise ValueError('Cannot normalize a zero or non-finite spectrum to its peak.')
    return 20 * np.log10(np.maximum(magnitude / peak, 1e-6))

# The originally disabled plot_rx_fft figure, with an additional dB view.
rx_freq = np.fft.fftshift(np.fft.fftfreq(len(rx), d=1 / Fs))
rx_fft = np.abs(np.fft.fftshift(np.fft.fft(rx))) / len(rx)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(rx_freq / 1e6, rx_fft)
axes[0].set(title='Received signal FFT', ylabel='FFT magnitude / N')
axes[1].plot(rx_freq / 1e6, relative_db(rx_fft))
axes[1].set(xlabel='Baseband frequency (MHz)', ylabel='Magnitude (dB relative to peak)')
for ax in axes:
    ax.grid(True)
fig.tight_layout()
plt.show()

# The originally disabled plot_rx_spectogram figure.
spec_freq, spec_time, spec_mag = spectrogram(
    rx, fs=Fs, window='hann', nperseg=256, noverlap=224,
    detrend=False, return_onesided=False, scaling='spectrum', mode='magnitude',
)
spec_freq = np.fft.fftshift(spec_freq)
spec_mag = np.fft.fftshift(spec_mag, axes=0)
fig, ax = plt.subplots(figsize=(10, 4))
image = ax.pcolormesh(spec_time * 1e6, spec_freq / 1e6, relative_db(spec_mag),
                      shading='auto', vmin=-60, vmax=0)
ax.set(xlabel='Time (us)', ylabel='Baseband frequency (MHz)', title='Received signal spectrogram')
fig.colorbar(image, ax=ax, label='Magnitude (dB relative to peak)')
fig.tight_layout()
plt.show()

## Matched filtering and range

A matched filter uses the time-reversed complex conjugate of the transmitted pulse. Its convolution with the received signal is correlation:

$$h[n]=s_{\mathrm{TX}}^*[N-1-n],\qquad y=r*h,\qquad R=\frac{c\tau}{2}.$$

For full convolution, zero lag occurs at index $N-1$. We discard negative lags and convert the remaining sample delays to ranges. The peak is narrow compared with the transmitted pulse; sidelobes are not necessarily targets.

![Example compressed pulse and sidelobes](Compress/3.jpeg)

![Illustrative matched-filter pulse compression](Compress/6.jpeg)

In [ ]:
matched_filter = np.conj(tx[::-1])
compressed = fftconvolve(rx, matched_filter, mode='full')
lags = np.arange(len(compressed)) - (N - 1)
valid = lags >= 0
range_axis = lags[valid] / Fs * c / 2
range_profile = np.abs(compressed[valid])

# Preserve the script's peak settings; distance is in samples, not meters.
peaks, _ = find_peaks(range_profile, distance=20, prominence=0.10 * np.max(range_profile))
print(f'Peak minimum separation: {20 * c / (2 * Fs):g} m')
print('Pulsed estimates (m):', np.round(range_axis[peaks], 2))

# The originally disabled plot_compressed_and_targets figure.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range_axis, range_profile, label='Matched-filter magnitude')
for target in targets:
    ax.axvline(target, color='red', linestyle='--', alpha=0.5)
ax.set(xlabel='Range (m)', ylabel='Correlation magnitude', title='Pulse compression: targets vs estimates')
ax.grid(True)
fig.tight_layout()
plt.show()

## FMCW dechirping

For a stationary target, the simplified baseband echo is $s_{\mathrm{RX}}(t)=A e^{j\pi S(t-\tau)^2}$. During the TX/RX overlap, the script mixes TX with the conjugated RX:

$$s_{\mathrm{IF}}(t)=s_{\mathrm{TX}}(t)s_{\mathrm{RX}}^*(t)=A^*e^{j2\pi S\tau t}e^{-j\pi S\tau^2}.$$

Thus the beat frequency is $f_b=S\tau$ and $R=cf_b/(2S)$. The script uses continuous delays for this FMCW model, unlike the integer-sample delays in its pulsed model. Carrier propagation phase and Doppler are not modeled.

![Delayed TX and RX sweeps](Compress/1.jpeg)

![Sweep and beat-frequency example](Compress/2.jpeg)

![TX and RX waveform timing](Compress/5.jpeg)

![Beat-frequency and range-spectrum sketch](Compress/8.jpeg)

![Schematic of TX, RX and beat-frequency separation](Compress/a.svg)

These figures are conceptual and are not necessarily drawn to scale. For a stationary target with the same slope on TX and RX, their ideal frequency separation is constant.

In [ ]:
rx_fmcw = np.zeros(N, dtype=complex)
for distance, amplitude in zip(targets, amplitudes):
    tau = 2 * distance / c
    delayed_phase = np.exp(1j * np.pi * S * (t - tau)**2)
    valid_echo = (t >= tau) & (t < T + tau)
    rx_fmcw[valid_echo] += amplitude * delayed_phase[valid_echo]

# Use the overlap where all echoes and the transmit reference are present.
valid_time = t >= delays.max()
if not np.any(valid_time):
    raise ValueError('The target delays leave no TX/RX overlap in this chirp.')
if_signal = tx[valid_time] * np.conj(rx_fmcw[valid_time])
expected_beats = S * delays
for distance, beat in zip(targets, expected_beats):
    print(f'{distance:g} m: expected beat {beat / 1e6:.3f} MHz')

## FMCW spectrum and range

A Hann window reduces sidelobes for typical non-bin-aligned tones, at the cost of a wider mainlobe and changed amplitude. The FFT measures beat frequencies, which are mapped to range using $R=cf_b/(2S)$. Zero-padding makes the displayed frequency grid denser; it does **not** improve physical resolution.

![Illustrative bin-aligned signal with and without a Hann window](Compress/4.jpeg)

In [ ]:
window = np.hanning(len(if_signal))
NFFT = max(65536, len(if_signal))
fft_values = np.fft.fft(if_signal * window, n=NFFT)
fft_frequencies = np.fft.fftfreq(NFFT, d=1 / Fs)
positive = fft_frequencies >= 0
frequencies = fft_frequencies[positive]
spectrum = np.abs(fft_values[positive])
range_if = c * frequencies / (2 * S)

# Detection remains on linear magnitude, as in SignalDetect.py.
if_peaks, _ = find_peaks(spectrum, distance=100, prominence=0.10 * np.max(spectrum))
print(f'FFT grid spacing: {Fs / NFFT / 1e3:.3f} kHz')
print(f'Peak minimum separation: {100 * c * Fs / (2 * S * NFFT):.2f} m')
for p in if_peaks:
    print(f'Estimated range: {range_if[p]:.2f} m; beat: {frequencies[p] / 1e6:.3f} MHz')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(frequencies / 1e6, spectrum)
axes[0].set(title='FMCW IF spectrum', ylabel='FFT magnitude')
axes[1].plot(frequencies / 1e6, relative_db(spectrum))
axes[1].set(xlabel='Beat frequency (MHz)', ylabel='Magnitude (dB relative to peak)')
axes[1].set_xlim(0, max(5, 1.1 * np.max(expected_beats) / 1e6))
for ax in axes:
    ax.grid(True)
fig.tight_layout()
plt.show()

## Comparison and limitations

The two range profiles below reproduce the script's final comparison, with additional relative-dB views. Each profile is normalized independently, so their dB levels should not be compared as absolute received power.

The script's peak-finder `distance` and `prominence` settings can suppress nearby or weak targets independently of physical resolution. Coherent echo phase, integer delay rounding, windowing and SNR also affect the apparent peaks.

A sequence of chirps can be used for Doppler processing; the following illustrative range-Doppler diagram is **not** computed by this single-chirp notebook.

![Illustration of repeated chirps and a range-Doppler map](Compress/7.jpeg)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes[0, 0].plot(range_axis, range_profile)
axes[0, 0].set(title='Pulsed LFM: matched filter', ylabel='Correlation magnitude')
axes[0, 1].plot(range_axis, relative_db(range_profile))
axes[0, 1].set(title='Pulsed LFM: relative dB', ylabel='Magnitude (dB relative to peak)')
axes[1, 0].plot(range_if, spectrum)
axes[1, 0].set(title='FMCW: IF FFT', xlabel='Range (m)', ylabel='FFT magnitude')
axes[1, 1].plot(range_if, relative_db(spectrum))
axes[1, 1].set(title='FMCW: relative dB', xlabel='Range (m)', ylabel='Magnitude (dB relative to peak)')
for ax in axes.flat:
    ax.set_xlim(0, max(150, 1.1 * np.max(targets)))
    ax.grid(True)
    for target in targets:
        ax.axvline(target, color='red', linestyle='--', alpha=0.5)
fig.tight_layout()
plt.show()